[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Fixtures &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the project's folders and `run_pytest`, and the cells after it write the
module, the summary script and `tests/conftest.py` as the notebook left them. Run them first, then
the tasks in order, since every task builds on the one before it. The last cell removes the scratch
folder.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


print("ready:", PROJECT)


ready: scratch/stations


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


In [3]:
%%writefile scratch/stations/summary.py
"""Print each station's mean temperature from a file of readings.

    python summary.py readings.csv

With no path given, the path comes from the environment variable STATIONS_READINGS.
"""

import os
import sys

from readings import summarize, to_fahrenheit


def main(arguments):
    path = arguments[0] if arguments else os.environ["STATIONS_READINGS"]
    with open(path, encoding="utf-8") as file:
        summary = summarize(file)
    for station, celsius in summary.items():
        if celsius is None:
            print(station, "no readings")
        else:
            print(station, f"{celsius:.1f} C, {to_fahrenheit(celsius):.1f} F")


if __name__ == "__main__":
    main(sys.argv[1:])


Writing scratch/stations/summary.py


In [4]:
%%writefile scratch/stations/tests/conftest.py
import pytest


@pytest.fixture
def tuesday():
    """Tuesday's lines of readings, as the stations sent them."""
    return ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
            "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


@pytest.fixture
def tuesday_file(tmp_path, tuesday):
    """Tuesday's readings, in a file of their own."""
    path = tmp_path / "tuesday.csv"
    path.write_text("\n".join(tuesday) + "\n", encoding="utf-8")
    return path


Writing scratch/stations/tests/conftest.py


**1.** A fixture of Oslo's readings.


In [5]:
%%writefile scratch/stations/tests/test_oslo.py
import pytest

from readings import summarize


@pytest.fixture
def oslo_lines():
    return ["Oslo,-2.4", "Oslo,-1.6"]


def test_oslo_summarizes(oslo_lines):
    assert summarize(oslo_lines) == {"Oslo": -2.0}


Writing scratch/stations/tests/test_oslo.py


In [6]:
run_pytest("tests/test_oslo.py", "-v")


============================= test session starts ==============================
collecting ... collected 1 item

tests/test_oslo.py::test_oslo_summarizes PASSED                          [100%]

============================== 1 passed ===============================


The test names `oslo_lines`, and never calls it: pytest does.


**2.** A second test, and how often the fixture ran.


In [7]:
%%writefile scratch/stations/tests/test_oslo.py
import pytest

from readings import summarize


@pytest.fixture
def oslo_lines():
    return ["Oslo,-2.4", "Oslo,-1.6"]


def test_oslo_summarizes(oslo_lines):
    assert summarize(oslo_lines) == {"Oslo": -2.0}


def test_oslo_sent_two_lines(oslo_lines):
    assert len(oslo_lines) == 2


Overwriting scratch/stations/tests/test_oslo.py


In [8]:
report = pytest_report("tests/test_oslo.py", "--setup-show", "-q")

print(report)
print("\noslo_lines was set up", report.count("SETUP    F oslo_lines"), "times")



        SETUP    F oslo_lines
        tests/test_oslo.py::test_oslo_summarizes (fixtures used: oslo_lines).
        TEARDOWN F oslo_lines
        SETUP    F oslo_lines
        tests/test_oslo.py::test_oslo_sent_two_lines (fixtures used: oslo_lines).
        TEARDOWN F oslo_lines
2 passed

oslo_lines was set up 2 times


Twice, once for each test that requested it, since its scope is a function.


**3.** A file built from the lines.


In [9]:
%%writefile scratch/stations/tests/test_oslo.py
import pytest

from readings import summarize


@pytest.fixture
def oslo_lines():
    return ["Oslo,-2.4", "Oslo,-1.6"]


@pytest.fixture
def oslo_file(tmp_path, oslo_lines):
    path = tmp_path / "oslo.csv"
    path.write_text("\n".join(oslo_lines) + "\n", encoding="utf-8")
    return path


def test_oslo_summarizes(oslo_lines):
    assert summarize(oslo_lines) == {"Oslo": -2.0}


def test_oslo_sent_two_lines(oslo_lines):
    assert len(oslo_lines) == 2


def test_the_oslo_file_summarizes(oslo_file):
    with open(oslo_file, encoding="utf-8") as file:
        assert summarize(file) == {"Oslo": -2.0}


Overwriting scratch/stations/tests/test_oslo.py


In [10]:
run_pytest("tests/test_oslo.py", "-q")


...                                                                      [100%]
3 passed


`oslo_file` requests `oslo_lines` and `tmp_path`, as a test would, and pytest sets both up first.


**4.** The fixtures, moved into conftest.py.


In [11]:
conftest = PROJECT / "tests" / "conftest.py"
conftest.write_text(conftest.read_text() + '''

@pytest.fixture
def oslo_lines():
    return ["Oslo,-2.4", "Oslo,-1.6"]


@pytest.fixture
def oslo_file(tmp_path, oslo_lines):
    path = tmp_path / "oslo.csv"
    path.write_text("\\n".join(oslo_lines) + "\\n", encoding="utf-8")
    return path
''')

tests = PROJECT / "tests" / "test_oslo.py"
source = tests.read_text()
tests.write_text("from readings import summarize\n\n\n" + source[source.index("def test_"):])

run_pytest("tests/test_oslo.py", "-q")


...                                                                      [100%]
3 passed


The test file lost its fixtures and its `import pytest`, and its tests still found both fixtures, in
`conftest.py`. Inside the text written to `conftest.py`, `\\n` stands for the two characters `\n`, so
that the file holds the same code as task 3.


**5.** capsys, on main.


In [12]:
%%writefile scratch/stations/tests/test_oslo_script.py
from summary import main


def test_main_prints_oslo(oslo_file, capsys):
    main([str(oslo_file)])

    assert capsys.readouterr().out == "Oslo -2.0 C, 28.4 F\n"


Writing scratch/stations/tests/test_oslo_script.py


In [13]:
run_pytest("tests/test_oslo_script.py", "-q")


.                                                                        [100%]
1 passed


`readouterr` returned everything `main` printed, newline included.


**6.** monkeypatch, and the notebook's own environment.


In [14]:
%%writefile scratch/stations/tests/test_oslo_environment.py
from summary import main


def test_main_reads_oslo_from_the_environment(oslo_file, capsys, monkeypatch):
    monkeypatch.setenv("STATIONS_READINGS", str(oslo_file))

    main([])

    assert capsys.readouterr().out == "Oslo -2.0 C, 28.4 F\n"


Writing scratch/stations/tests/test_oslo_environment.py


In [15]:
run_pytest("tests/test_oslo_environment.py", "-q")

print("\nSTATIONS_READINGS set in this notebook:", "STATIONS_READINGS" in os.environ)


.                                                                        [100%]
1 passed

STATIONS_READINGS set in this notebook: False


The variable was set inside the pytest process, which is a program of its own, and `monkeypatch` put
that process's environment back after the test. The notebook's environment was never touched.

Last, remove the scratch folder:


In [16]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Fixtures](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/04-fixtures.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
